In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

columns = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_text", "comments",
    "date_posted", "latitude", "longitude"
]

df = pd.read_csv(url, names=columns, header=None)

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df = df.dropna(subset=["datetime"])

df["year"] = df["datetime"].dt.year

df.head()

In [ ]:
df_year = df.groupby('year').size().reset_index(name='sightings')

plot1 = alt.Chart(df_year).mark_line(point=True).encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('sightings:Q', title='Number of Sightings'),
    tooltip=['year:O', 'sightings:Q']
).properties(
    title='UFO Sightings Over Time',
    width=700,
    height=400
)

plot1

In [ ]:
df2 = df.dropna(subset=['shape']).copy()
df2['shape'] = df2['shape'].str.lower()

top_shapes = df2['shape'].value_counts().nlargest(10).index.tolist()
df2 = df2[df2['shape'].isin(top_shapes)]

In [ ]:
year_list = sorted(df2['year'].unique())

dropdown = alt.binding_select(options=year_list, name='Select year: ')
selection = alt.selection_point(fields=['year'], bind=dropdown, value=2000)

In [ ]:
plot2 = alt.Chart(df2).transform_aggregate(
    sightings='count()',
    groupby=['year', 'shape']
).mark_bar().encode(
    x=alt.X('shape:N', sort='-y', title='Shape'),
    y=alt.Y('sightings:Q', title='Sightings'),
    color=alt.Color('shape:N'),
    tooltip=['shape:N', 'sightings:Q']
).add_params(
    selection
).transform_filter(
    selection
).properties(
    title='Top UFO Shapes for Selected Year',
    width=700,
    height=400
)

plot2

In [ ]:
plot1.save('ufo_plot1.json')
plot2.save('ufo_plot2.json')

In [ ]:
import os
print(os.listdir())

In [ ]:
from google.colab import files
files.download('ufo_plot1.json')
files.download('ufo_plot2.json')